# 04 - Multiple Tool Calls & Parallel Execution

This notebook explores advanced function calling patterns where the AI can execute multiple functions **simultaneously** in a single request.

## 🎯 What You'll Learn
- **Parallel Execution**: Multiple functions called at once
- **Complex Workflows**: Building sophisticated AI workflows
- **Efficiency Gains**: Reducing API calls and latency
- **Real-world Scenarios**: Practical applications of parallel tool calling

## 🚀 Key Benefits of Multiple Tool Calls
- ⚡ **Faster Response Times**: Parallel execution vs sequential
- 💰 **Cost Efficiency**: Fewer API round-trips
- 🧠 **Smarter Workflows**: AI can plan complex multi-step operations
- 🔄 **Better User Experience**: Single request handles complex queries

In [1]:
import os
import json
import time
from datetime import datetime
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize client
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT4O_API_VERSION")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version,
)

print("✅ Multiple tool calls setup complete!")
print("🚀 Ready to demonstrate parallel execution!")

✅ Multiple tool calls setup complete!
🚀 Ready to demonstrate parallel execution!


## 2 - Multiple Tool Definitions

Let's create a suite of tools that work well together:

In [2]:
# Define multiple tools for different purposes
multiple_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_time_zone",
            "description": "Get the current time and timezone information for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_travel_info",
            "description": "Get travel information between two locations",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_location": {
                        "type": "string",
                        "description": "Starting location"
                    },
                    "to_location": {
                        "type": "string",
                        "description": "Destination location"
                    },
                    "travel_mode": {
                        "type": "string",
                        "enum": ["driving", "flying", "train"],
                        "description": "Mode of transportation"
                    }
                },
                "required": ["from_location", "to_location"]
            }
        }
    }
]

print(f"🛠️ Defined {len(multiple_tools)} tools:")
for tool in multiple_tools:
    print(f"   📞 {tool['function']['name']}")
print("\n✨ Ready for simultaneous calls!")

🛠️ Defined 3 tools:
   📞 get_current_weather
   📞 get_time_zone
   📞 get_travel_info

✨ Ready for simultaneous calls!


## 3 - Complex Query Requiring Multiple Tools

Let's ask a question that naturally requires multiple tools to answer completely:

In [3]:
# Complex query that benefits from multiple tool calls
complex_query = [
    {
        "role": "user",
        "content": "I'm planning a trip from New York to Tokyo. Can you tell me the weather in both cities, what time it is there, and how long it takes to fly between them?"
    }
]

print("❓ Complex Query:")
print(f"   {complex_query[0]['content']}")
print("\n🤔 This requires multiple tools:")
print("   🌤️ Weather in New York")
print("   🌤️ Weather in Tokyo")
print("   🕐 Time in New York")
print("   🕐 Time in Tokyo")
print("   ✈️ Flight information")
print("\n🚀 Let's see how AI handles this!")

❓ Complex Query:
   I'm planning a trip from New York to Tokyo. Can you tell me the weather in both cities, what time it is there, and how long it takes to fly between them?

🤔 This requires multiple tools:
   🌤️ Weather in New York
   🌤️ Weather in Tokyo
   🕐 Time in New York
   🕐 Time in Tokyo
   ✈️ Flight information

🚀 Let's see how AI handles this!


In [4]:
# Make the API call - AI will determine which tools to use
start_time = time.time()

multi_completion = client.chat.completions.create(
    model=deployment,
    messages=complex_query,
    tools=multiple_tools,
    tool_choice="auto",  # Let AI decide which tools to use
    max_tokens=1000,
    temperature=0.3  # Lower temperature for more focused tool selection
)

api_call_time = time.time() - start_time

print(f"⚡ API call completed in {api_call_time:.2f} seconds")
print("\n📞 MULTIPLE TOOL CALLS RESULT:")
print("=" * 60)

message = multi_completion.choices[0].message
print(f"Role: {message.role}")
print(f"Content: {message.content}")
print(f"Tool Calls: {len(message.tool_calls) if message.tool_calls else 0} calls")
print("=" * 60)

⚡ API call completed in 4.11 seconds

📞 MULTIPLE TOOL CALLS RESULT:
Role: assistant
Content: None
Tool Calls: 5 calls


## 4 - Multiple Tool Calls Analysis

Let's examine what the AI decided to call:

In [5]:
# Analyze all the tool calls the AI made
if hasattr(message, 'tool_calls') and message.tool_calls:
    tool_calls = message.tool_calls
    print(f"🎯 AI MADE {len(tool_calls)} SIMULTANEOUS TOOL CALLS:")
    print("=" * 60)
    
    for i, tool_call in enumerate(tool_calls, 1):
        print(f"\n📞 Tool Call #{i}:")
        print(f"   🆔 ID: {tool_call.id}")
        print(f"   🎯 Function: {tool_call.function.name}")
        
        try:
            args = json.loads(tool_call.function.arguments)
            print(f"   📋 Arguments: {args}")
        except json.JSONDecodeError:
            print(f"   📋 Arguments (raw): {tool_call.function.arguments}")
    
    print(f"\n✨ All {len(tool_calls)} calls planned in a SINGLE API request!")
    print("🚀 This is the power of modern function calling!")
else:
    print("❌ No tool calls detected")

🎯 AI MADE 5 SIMULTANEOUS TOOL CALLS:

📞 Tool Call #1:
   🆔 ID: call_14hh2LAoln9KSLiu5fpB7xJR
   🎯 Function: get_current_weather
   📋 Arguments: {'location': 'New York', 'unit': 'celsius'}

📞 Tool Call #2:
   🆔 ID: call_jT30MQkp5NDkV5wzpQ2q5KEj
   🎯 Function: get_current_weather
   📋 Arguments: {'location': 'Tokyo', 'unit': 'celsius'}

📞 Tool Call #3:
   🆔 ID: call_9TMtKy5Ed9QUAscJazqiZtG0
   🎯 Function: get_time_zone
   📋 Arguments: {'location': 'New York'}

📞 Tool Call #4:
   🆔 ID: call_5vX6rE6pSLQA3bSvfPvB0RjA
   🎯 Function: get_time_zone
   📋 Arguments: {'location': 'Tokyo'}

📞 Tool Call #5:
   🆔 ID: call_lXttec8iv265zMEyg5fkfpSy
   🎯 Function: get_travel_info
   📋 Arguments: {'from_location': 'New York', 'to_location': 'Tokyo', 'travel_mode': 'flying'}

✨ All 5 calls planned in a SINGLE API request!
🚀 This is the power of modern function calling!


## 5 - Executing Multiple Tools

Now let's implement and execute all the tools the AI requested:

In [6]:
# Tool implementations
def get_current_weather(location, unit="celsius"):
    """Simulate weather API calls"""
    # Simulate different weather for different cities
    weather_db = {
        "New York": {
            "temperature": "22" if unit == "celsius" else "72",
            "description": "Sunny with light clouds",
            "humidity": "65%",
            "wind_speed": "15 km/h"
        },
        "Tokyo": {
            "temperature": "28" if unit == "celsius" else "82",
            "description": "Hot and humid",
            "humidity": "85%",
            "wind_speed": "8 km/h"
        }
    }
    
    # Find matching city
    city_key = None
    for city in weather_db.keys():
        if city.lower() in location.lower():
            city_key = city
            break
    
    if city_key:
        weather = weather_db[city_key].copy()
        weather["location"] = location
        weather["unit"] = unit
        weather["timestamp"] = datetime.now().isoformat()
    else:
        weather = {
            "location": location,
            "temperature": "20" if unit == "celsius" else "68",
            "unit": unit,
            "description": "Partly cloudy",
            "humidity": "70%",
            "wind_speed": "10 km/h",
            "timestamp": datetime.now().isoformat()
        }
    
    return json.dumps(weather)

def get_time_zone(location):
    """Simulate timezone API calls"""
    timezone_db = {
        "New York": {
            "timezone": "America/New_York",
            "current_time": "2024-01-15 09:30:00",
            "utc_offset": "-05:00",
            "is_dst": False
        },
        "Tokyo": {
            "timezone": "Asia/Tokyo",
            "current_time": "2024-01-15 23:30:00",
            "utc_offset": "+09:00",
            "is_dst": False
        }
    }
    
    # Find matching city
    city_key = None
    for city in timezone_db.keys():
        if city.lower() in location.lower():
            city_key = city
            break
    
    if city_key:
        time_info = timezone_db[city_key].copy()
        time_info["location"] = location
    else:
        time_info = {
            "location": location,
            "timezone": "UTC",
            "current_time": "2024-01-15 14:30:00",
            "utc_offset": "+00:00",
            "is_dst": False
        }
    
    return json.dumps(time_info)

def get_travel_info(from_location, to_location, travel_mode="flying"):
    """Simulate travel API calls"""
    travel_info = {
        "from": from_location,
        "to": to_location,
        "mode": travel_mode,
        "duration": "14 hours 30 minutes" if travel_mode == "flying" else "Not available",
        "distance": "10,850 km (6,740 miles)",
        "estimated_cost": "$800-2500 USD" if travel_mode == "flying" else "Not available",
        "notes": "Direct flights available" if travel_mode == "flying" else "No direct route"
    }
    
    return json.dumps(travel_info)

print("🔧 Tool implementations ready!")
print("📞 get_current_weather() - Simulates weather API")
print("🕐 get_time_zone() - Simulates timezone API")
print("✈️ get_travel_info() - Simulates travel API")

🔧 Tool implementations ready!
📞 get_current_weather() - Simulates weather API
🕐 get_time_zone() - Simulates timezone API
✈️ get_travel_info() - Simulates travel API


In [7]:
# Execute all tool calls in parallel simulation
if hasattr(message, 'tool_calls') and message.tool_calls:
    print("🚀 EXECUTING ALL TOOL CALLS:")
    print("=" * 50)
    
    # Add assistant's message with tool calls
    complex_query.append({
        "role": "assistant",
        "content": message.content,
        "tool_calls": [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in message.tool_calls
        ]
    })
    
    # Execute each tool call
    execution_start = time.time()
    
    for i, tool_call in enumerate(message.tool_calls, 1):
        print(f"\n🔧 Executing Tool #{i}: {tool_call.function.name}")
        print(f"   🆔 ID: {tool_call.id}")
        
        try:
            args = json.loads(tool_call.function.arguments)
            print(f"   📋 Arguments: {args}")
            
            # Execute the appropriate function
            if tool_call.function.name == "get_current_weather":
                result = get_current_weather(
                    location=args["location"],
                    unit=args.get("unit", "celsius")
                )
            elif tool_call.function.name == "get_time_zone":
                result = get_time_zone(location=args["location"])
            elif tool_call.function.name == "get_travel_info":
                result = get_travel_info(
                    from_location=args["from_location"],
                    to_location=args["to_location"],
                    travel_mode=args.get("travel_mode", "flying")
                )
            else:
                result = json.dumps({"error": "Unknown function"})
            
            # Add tool result to conversation
            complex_query.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
            
            print(f"   ✅ Success: {len(result)} characters returned")
            
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
            complex_query.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({"error": str(e)})
            })
    
    execution_time = time.time() - execution_start
    print(f"\n⚡ All tools executed in {execution_time:.2f} seconds")
    print(f"🎯 Total tools processed: {len(message.tool_calls)}")
    
else:
    print("❌ No tool calls to execute")

🚀 EXECUTING ALL TOOL CALLS:

🔧 Executing Tool #1: get_current_weather
   🆔 ID: call_14hh2LAoln9KSLiu5fpB7xJR
   📋 Arguments: {'location': 'New York', 'unit': 'celsius'}
   ✅ Success: 193 characters returned

🔧 Executing Tool #2: get_current_weather
   🆔 ID: call_jT30MQkp5NDkV5wzpQ2q5KEj
   📋 Arguments: {'location': 'Tokyo', 'unit': 'celsius'}
   ✅ Success: 179 characters returned

🔧 Executing Tool #3: get_time_zone
   🆔 ID: call_9TMtKy5Ed9QUAscJazqiZtG0
   📋 Arguments: {'location': 'New York'}
   ✅ Success: 136 characters returned

🔧 Executing Tool #4: get_time_zone
   🆔 ID: call_5vX6rE6pSLQA3bSvfPvB0RjA
   📋 Arguments: {'location': 'Tokyo'}
   ✅ Success: 127 characters returned

🔧 Executing Tool #5: get_travel_info
   🆔 ID: call_lXttec8iv265zMEyg5fkfpSy
   📋 Arguments: {'from_location': 'New York', 'to_location': 'Tokyo', 'travel_mode': 'flying'}
   ✅ Success: 199 characters returned

⚡ All tools executed in 0.00 seconds
🎯 Total tools processed: 5


## 6 - Final Response with All Tool Results

Now let's get the AI's final response that combines all the tool results:

In [8]:
# Get the final comprehensive response
final_start = time.time()

final_completion = client.chat.completions.create(
    model=deployment,
    messages=complex_query,
    tools=multiple_tools,
    max_tokens=1200,
    temperature=0.7
)

final_time = time.time() - final_start
total_time = api_call_time + execution_time + final_time

print("🎯 FINAL COMPREHENSIVE RESPONSE:")
print("=" * 70)
print(final_completion.choices[0].message.content)
print("=" * 70)

print("\n⚡ PERFORMANCE ANALYSIS:")
print(f"   🚀 Initial API call: {api_call_time:.2f}s")
print(f"   🔧 Tool execution: {execution_time:.2f}s")
print(f"   📝 Final response: {final_time:.2f}s")
print(f"   🏁 Total time: {total_time:.2f}s")

print("\n💰 TOKEN USAGE SUMMARY:")
print(f"   🥇 First call: {multi_completion.usage.total_tokens} tokens")
print(f"   🥈 Final call: {final_completion.usage.total_tokens} tokens")
print(f"   🔢 Total tokens: {multi_completion.usage.total_tokens + final_completion.usage.total_tokens}")

🎯 FINAL COMPREHENSIVE RESPONSE:
Here is the information for your trip from New York to Tokyo:

### Weather
- **New York**: Sunny with light clouds, 22°C, 65% humidity, wind speed of 15 km/h.
- **Tokyo**: Hot and humid, 28°C, 85% humidity, wind speed of 8 km/h.

### Current Time
- **New York**: 09:30 AM (Time Zone: America/New_York, UTC Offset: -05:00)
- **Tokyo**: 11:30 PM (Time Zone: Asia/Tokyo, UTC Offset: +09:00)

### Flight Information
- **Duration**: Approximately 14 hours and 30 minutes.
- **Distance**: 10,850 km (6,740 miles).
- **Estimated Cost**: $800–$2500 USD.
- **Notes**: Direct flights are available.

Let me know if you'd like further assistance with your trip planning!

⚡ PERFORMANCE ANALYSIS:
   🚀 Initial API call: 4.11s
   🔧 Tool execution: 0.00s
   📝 Final response: 3.29s
   🏁 Total time: 7.40s

💰 TOKEN USAGE SUMMARY:
   🥇 First call: 325 tokens
   🥈 Final call: 824 tokens
   🔢 Total tokens: 1149


## 7 - Multiple Tool Calls vs Legacy Comparison

Let's analyze what we accomplished:

In [9]:
# Calculate what this would have cost with legacy functions
if hasattr(message, 'tool_calls') and message.tool_calls:
    num_tools = len(message.tool_calls)
    
    print("📊 EFFICIENCY COMPARISON:")
    print("=" * 50)
    
    print(f"🆕 MODERN APPROACH (What we did):")
    print(f"   📞 API calls made: 2 (initial + final)")
    print(f"   🛠️ Tools executed: {num_tools} (simultaneously)")
    print(f"   ⚡ Total time: {total_time:.2f} seconds")
    print(f"   💰 Total tokens: {multi_completion.usage.total_tokens + final_completion.usage.total_tokens}")
    
    print(f"\n🏛️ LEGACY APPROACH (Hypothetical):")
    # Legacy formula: 1 initial call + (each tool requires 2 calls: function call + response)
    legacy_calls = 1 + (num_tools * 2)  # 1 + (5 × 2) = 11 calls
    legacy_time = total_time * (legacy_calls / 2)  # Rough estimate
    legacy_tokens = (multi_completion.usage.total_tokens + final_completion.usage.total_tokens) * 1.5  # Rough estimate
    
    print(f"   📞 API calls needed: {legacy_calls}")
    print(f"   🛠️ Tools executed: {num_tools} (sequentially)")
    print(f"   ⚡ Estimated time: {legacy_time:.2f} seconds")
    print(f"   💰 Estimated tokens: {legacy_tokens:.0f}")
    
    print(f"\n📋 DETAILED LEGACY CALL SEQUENCE:")
    print(f"   📞 Call 1: User asks complex question")
    print(f"   🤖 Call 2: AI responds with FIRST function call (weather NYC)")
    print(f"   📊 Call 3: System returns NYC weather → AI processes")
    print(f"   🤖 Call 4: AI makes SECOND function call (weather Tokyo)")
    print(f"   📊 Call 5: System returns Tokyo weather → AI processes")
    print(f"   🤖 Call 6: AI makes THIRD function call (timezone NYC)")
    print(f"   📊 Call 7: System returns NYC timezone → AI processes")
    print(f"   🤖 Call 8: AI makes FOURTH function call (timezone Tokyo)")
    print(f"   📊 Call 9: System returns Tokyo timezone → AI processes")
    print(f"   🤖 Call 10: AI makes FIFTH function call (travel info)")
    print(f"   📊 Call 11: System returns travel data → AI synthesizes final answer")
    print(f"   🔄 Notice: Each tool needs 2 calls (AI request + system response)")
    
    print(f"\n🚀 MODERN ADVANTAGES:")
    print(f"   📞 {legacy_calls - 2} fewer API calls")
    print(f"   ⚡ {legacy_time - total_time:.2f}s faster")
    print(f"   💰 {legacy_tokens - (multi_completion.usage.total_tokens + final_completion.usage.total_tokens):.0f} fewer tokens")
    print(f"   🎯 {((legacy_calls - 2) / legacy_calls * 100):.1f}% reduction in API calls")
    
    print("\n✨ This is why modern function calling is revolutionary!")
    print("   🚀 Modern: Plan ALL tools → Execute ALL → Synthesize (2 calls)")
    print("   🐌 Legacy: Plan 1 → Execute 1 → Plan 1 → Execute 1... (11 calls)")
else:
    print("❌ No tool calls to analyze")

📊 EFFICIENCY COMPARISON:
🆕 MODERN APPROACH (What we did):
   📞 API calls made: 2 (initial + final)
   🛠️ Tools executed: 5 (simultaneously)
   ⚡ Total time: 7.40 seconds
   💰 Total tokens: 1149

🏛️ LEGACY APPROACH (Hypothetical):
   📞 API calls needed: 11
   🛠️ Tools executed: 5 (sequentially)
   ⚡ Estimated time: 40.71 seconds
   💰 Estimated tokens: 1724

📋 DETAILED LEGACY CALL SEQUENCE:
   📞 Call 1: User asks complex question
   🤖 Call 2: AI responds with FIRST function call (weather NYC)
   📊 Call 3: System returns NYC weather → AI processes
   🤖 Call 4: AI makes SECOND function call (weather Tokyo)
   📊 Call 5: System returns Tokyo weather → AI processes
   🤖 Call 6: AI makes THIRD function call (timezone NYC)
   📊 Call 7: System returns NYC timezone → AI processes
   🤖 Call 8: AI makes FOURTH function call (timezone Tokyo)
   📊 Call 9: System returns Tokyo timezone → AI processes
   🤖 Call 10: AI makes FIFTH function call (travel info)
   📊 Call 11: System returns travel data → AI

### 8 - Why 6 API Calls in Legacy Format?

The legacy `functions` format has a critical limitation: **only ONE function per API call**.

**Modern vs Legacy Breakdown:**
- **Modern**: AI plans all 3 tools → Execute all 3 → Synthesize (2 calls total)
- **Legacy**: Plan 1 tool → Execute 1 → Plan next → Execute next... (6 calls total)

**Complete Legacy Call Sequence (6 calls):**

**📞 Call 1:** User asks: *"I need weather, time, and travel info for NYC/Tokyo"*

**🤖 Call 2:** AI responds: *"I'll get NYC weather first"* → `get_weather("New York")`

**📊 Call 3:** System returns NYC weather data → AI processes and plans next

**🤖 Call 4:** AI responds: *"Getting NYC timezone"* → `get_time_zone("New York")`

**📊 Call 5:** System returns NYC time data → AI processes and plans next

**🤖 Call 6:** AI responds: *"Getting travel info"* → `get_travel_info("NYC", "Tokyo")`

**Formula: `1 initial + (2 tools × 2 calls each) + 1 call for travel info = 1 + 4 + 1 = 6 calls`**

**💡 Key Insight:** The third tool (`get_travel_info`) can handle both cities in a single call, reducing the total number of calls in the legacy format!

## 9 - Complete Conversation Flow Analysis

Let's examine the entire conversation structure:

In [10]:
print("📋 COMPLETE MULTI-TOOL CONVERSATION FLOW:")
print("=" * 60)

for i, msg in enumerate(complex_query, 1):
    print(f"\n📝 Message #{i}: {msg['role'].upper()}")
    
    if msg['role'] == 'user':
        print(f"   💬 Asked: {msg['content'][:60]}...")
    
    elif msg['role'] == 'assistant':
        if 'tool_calls' in msg and msg['tool_calls']:
            print(f"   🛠️ Requested {len(msg['tool_calls'])} tools:")
            for tc in msg['tool_calls']:
                print(f"      📞 {tc['id']}: {tc['function']['name']}")
        else:
            print(f"   💬 Responded: {msg.get('content', 'No content')[:60]}...")
    
    elif msg['role'] == 'tool':
        tool_result = json.loads(msg['content'])
        tool_name = "Unknown"
        # Find the tool name by ID
        for prev_msg in complex_query:
            if prev_msg['role'] == 'assistant' and 'tool_calls' in prev_msg:
                for tc in prev_msg['tool_calls']:
                    if tc['id'] == msg['tool_call_id']:
                        tool_name = tc['function']['name']
                        break
        
        print(f"   🔧 {msg['tool_call_id']}: {tool_name}")
        if 'location' in tool_result:
            print(f"      📍 Location: {tool_result['location']}")
        if 'temperature' in tool_result:
            print(f"      🌡️ Temperature: {tool_result['temperature']}°{tool_result.get('unit', 'C')}")
        if 'current_time' in tool_result:
            print(f"      🕐 Time: {tool_result['current_time']}")
        if 'duration' in tool_result:
            print(f"      ✈️ Duration: {tool_result['duration']}")

print("\n✨ Notice the clean, trackable flow with multiple simultaneous tools!")

📋 COMPLETE MULTI-TOOL CONVERSATION FLOW:

📝 Message #1: USER
   💬 Asked: I'm planning a trip from New York to Tokyo. Can you tell me ...

📝 Message #2: ASSISTANT
   🛠️ Requested 5 tools:
      📞 call_14hh2LAoln9KSLiu5fpB7xJR: get_current_weather
      📞 call_jT30MQkp5NDkV5wzpQ2q5KEj: get_current_weather
      📞 call_9TMtKy5Ed9QUAscJazqiZtG0: get_time_zone
      📞 call_5vX6rE6pSLQA3bSvfPvB0RjA: get_time_zone
      📞 call_lXttec8iv265zMEyg5fkfpSy: get_travel_info

📝 Message #3: TOOL
   🔧 call_14hh2LAoln9KSLiu5fpB7xJR: get_current_weather
      📍 Location: New York
      🌡️ Temperature: 22°celsius

📝 Message #4: TOOL
   🔧 call_jT30MQkp5NDkV5wzpQ2q5KEj: get_current_weather
      📍 Location: Tokyo
      🌡️ Temperature: 28°celsius

📝 Message #5: TOOL
   🔧 call_9TMtKy5Ed9QUAscJazqiZtG0: get_time_zone
      📍 Location: New York
      🕐 Time: 2024-01-15 09:30:00

📝 Message #6: TOOL
   🔧 call_5vX6rE6pSLQA3bSvfPvB0RjA: get_time_zone
      📍 Location: Tokyo
      🕐 Time: 2024-01-15 23:30:00

📝 Mes

## 10 - Advanced Multiple Tool Scenarios

Let's try one more example to show even more complex tool coordination:

In [11]:
# Even more complex scenario
super_complex_query = [
    {
        "role": "user",
        "content": "I need to compare weather in Paris, London, and Berlin. Also get the current time in each city and tell me the flight time from Paris to each of the other cities."
    }
]

print("🎯 SUPER COMPLEX QUERY:")
print(f"   {super_complex_query[0]['content']}")
print("\n🤔 This could require up to 8 tool calls:")
print("   🌤️ Weather: Paris, London, Berlin (3 calls)")
print("   🕐 Time: Paris, London, Berlin (3 calls)")
print("   ✈️ Travel: Paris→London, Paris→Berlin (2 calls)")
print("\n🚀 Let's see what AI actually decides to call...")

# Make the super complex call
super_completion = client.chat.completions.create(
    model=deployment,
    messages=super_complex_query,
    tools=multiple_tools,
    tool_choice="auto",
    max_tokens=1000,
    temperature=0.2
)

super_message = super_completion.choices[0].message

if hasattr(super_message, 'tool_calls') and super_message.tool_calls:
    print(f"\n🎯 AI MADE {len(super_message.tool_calls)} SIMULTANEOUS CALLS:")
    for i, tc in enumerate(super_message.tool_calls, 1):
        args = json.loads(tc.function.arguments)
        print(f"   📞 #{i}: {tc.function.name} - {args}")
    
    print(f"\n⚡ ALL {len(super_message.tool_calls)} PLANNED IN ONE API CALL!")
    print("🚀 This level of coordination is impossible with legacy functions!")
else:
    print("\n🤔 AI decided not to use tools, or there was an issue.")

🎯 SUPER COMPLEX QUERY:
   I need to compare weather in Paris, London, and Berlin. Also get the current time in each city and tell me the flight time from Paris to each of the other cities.

🤔 This could require up to 8 tool calls:
   🌤️ Weather: Paris, London, Berlin (3 calls)
   🕐 Time: Paris, London, Berlin (3 calls)
   ✈️ Travel: Paris→London, Paris→Berlin (2 calls)

🚀 Let's see what AI actually decides to call...

🎯 AI MADE 8 SIMULTANEOUS CALLS:
   📞 #1: get_current_weather - {'location': 'Paris', 'unit': 'celsius'}
   📞 #2: get_current_weather - {'location': 'London', 'unit': 'celsius'}
   📞 #3: get_current_weather - {'location': 'Berlin', 'unit': 'celsius'}
   📞 #4: get_time_zone - {'location': 'Paris'}
   📞 #5: get_time_zone - {'location': 'London'}
   📞 #6: get_time_zone - {'location': 'Berlin'}
   📞 #7: get_travel_info - {'from_location': 'Paris', 'to_location': 'London', 'travel_mode': 'flying'}
   📞 #8: get_travel_info - {'from_location': 'Paris', 'to_location': 'Berlin', 't

## 11 - Advanced Insights: tool_call_id Deep Dive

Now you can see why `tool_call_id` is so important:

In [12]:
if hasattr(super_message, 'tool_calls') and super_message.tool_calls:
    print("🔍 TOOL_CALL_ID IMPORTANCE DEMONSTRATION:")
    print("=" * 50)
    
    print("🆔 Each tool call gets a unique ID:")
    for i, tc in enumerate(super_message.tool_calls, 1):
        print(f"   #{i}: {tc.id} → {tc.function.name}")
    
    print("\n🔗 Why this matters:")
    print("   ✅ Each result can be linked back to its specific call")
    print("   ✅ AI can process results in any order")
    print("   ✅ Perfect for parallel/async execution")
    print("   ✅ Eliminates confusion with multiple similar calls")
    print("   ✅ Enables sophisticated error handling")
    
    print("\n🚫 Without tool_call_id:")
    print("   ❌ Can't tell which weather result belongs to which city")
    print("   ❌ Results could get mixed up")
    print("   ❌ No way to handle partial failures")
    print("   ❌ Debugging becomes nightmare")
    
    print("\n🎯 REAL EXAMPLE:")
    print("   If you call get_weather('Paris') and get_weather('London')")
    print("   simultaneously, tool_call_id tells you which result")
    print("   belongs to which city!")
    
    print("\n✨ This is why modern function calling is so powerful!")

🔍 TOOL_CALL_ID IMPORTANCE DEMONSTRATION:
🆔 Each tool call gets a unique ID:
   #1: call_IEOfKaHOHc8axIsXySJSKtbZ → get_current_weather
   #2: call_UCpCi3GU8f6KMhQdR5Ct3ibg → get_current_weather
   #3: call_sbJCxuBGgmDcdmxdGGU9m7wx → get_current_weather
   #4: call_qQhZbEnNqfCcocwhQg9wchkJ → get_time_zone
   #5: call_ANKppdqSttn1dIixtwUZVIxu → get_time_zone
   #6: call_8qEvdE8PuWymik68jTKZxwPN → get_time_zone
   #7: call_h616F5NdfjZNQXv8XS7gDKzH → get_travel_info
   #8: call_97J0PVU0xF2aLEXkKgptcJQY → get_travel_info

🔗 Why this matters:
   ✅ Each result can be linked back to its specific call
   ✅ AI can process results in any order
   ✅ Perfect for parallel/async execution
   ✅ Eliminates confusion with multiple similar calls
   ✅ Enables sophisticated error handling

🚫 Without tool_call_id:
   ❌ Can't tell which weather result belongs to which city
   ❌ Results could get mixed up
   ❌ No way to handle partial failures
   ❌ Debugging becomes nightmare

🎯 REAL EXAMPLE:
   If you call g

## 12 - Key Takeaways & Best Practices

### ✅ What You've Mastered
- **🚀 Multiple Simultaneous Calls**: The AI can plan and execute multiple tools at once
- **🆔 ID Tracking**: Every tool call has a unique identifier for perfect result linkage
- **⚡ Performance**: Dramatically faster and more efficient than legacy approach
- **🧠 Smart Coordination**: AI intelligently decides which tools to use together

### 🎯 Best Practices
1. **📋 Design Related Tools**: Create tools that work well together
2. **🆔 Always Use tool_call_id**: Essential for tracking results properly
3. **⚡ Embrace Parallel Thinking**: Ask questions that benefit from multiple tools
4. **🛡️ Handle Errors Gracefully**: Each tool can fail independently
5. **📊 Monitor Performance**: Track the efficiency gains

### 🚫 Common Pitfalls to Avoid
- ❌ Don't ignore tool_call_id in responses
- ❌ Don't assume tools will be called in any specific order
- ❌ Don't create tools that are too similar (causes confusion)
- ❌ Don't forget error handling for individual tools

---

## 🏆 Congratulations!

You've now mastered the most advanced aspects of Azure OpenAI function calling:

✅ **Basic Setup**: Understanding the foundation  
✅ **Legacy Format**: Historical context and limitations  
✅ **Modern Format**: Current best practices  
✅ **Multiple Tools**: Advanced parallel execution  

## 🚀 What's Next?

With this knowledge, you can:
- **🛠️ Build Complex AI Applications**: Create sophisticated tool ecosystems
- **⚡ Optimize Performance**: Use parallel tool calls for efficiency
- **🔗 Create Tool Chains**: Design workflows with multiple coordinated tools
- **📊 Build Real Applications**: Weather apps, travel planners, data analyzers

**You're now ready to build production-ready AI applications with function calling!** 🎯

In [13]:
# JSON structure of the full request being sent back to the LLM
full_request_to_llm = {
    "model": "gpt-4o-2024-11-20",
    "messages": [
        {
            "role": "user",
            "content": "I'm planning a trip from Amsterdam to London. Can you tell me the weather, time, and travel details?"
        }
    ],
    "tool_results": [
        {
            "tool_call_id": "call_Ra1i37ksFMyapOOpWgy4FYMk",
            "response": {
                "location": "Amsterdam",
                "temperature": "15",
                "unit": "celsius",
                "description": "Cloudy with light rain",
                "humidity": "80%",
                "wind_speed": "10 km/h"
            }
        },
        {
            "tool_call_id": "call_W0LWVzMt1OsVJ3VfSNZn87ZX",
            "response": {
                "timezone": "Europe/Amsterdam",
                "current_time": "2025-07-03T14:30:00",
                "utc_offset": "+02:00",
                "is_dst": True
            }
        },
        {
            "tool_call_id": "call_ExampleTravelInfo",
            "response": {
                "from_city": "Amsterdam",
                "to_city": "London",
                "travel_mode": "flight",
                "duration": "1 hour 15 minutes",
                "distance": "360 km",
                "estimated_cost": "$150-300 USD"
            }
        }
    ]
}

print("JSON structure of the full request being sent back to LLM:")
print(json.dumps(full_request_to_llm, indent=4))

JSON structure of the full request being sent back to LLM:
{
    "model": "gpt-4o-2024-11-20",
    "messages": [
        {
            "role": "user",
            "content": "I'm planning a trip from Amsterdam to London. Can you tell me the weather, time, and travel details?"
        }
    ],
    "tool_results": [
        {
            "tool_call_id": "call_Ra1i37ksFMyapOOpWgy4FYMk",
            "response": {
                "location": "Amsterdam",
                "temperature": "15",
                "unit": "celsius",
                "description": "Cloudy with light rain",
                "humidity": "80%",
                "wind_speed": "10 km/h"
            }
        },
        {
            "tool_call_id": "call_W0LWVzMt1OsVJ3VfSNZn87ZX",
            "response": {
                "timezone": "Europe/Amsterdam",
                "current_time": "2025-07-03T14:30:00",
                "utc_offset": "+02:00",
                "is_dst": true
            }
        },
        {
           